# MVP: риск-карта региональной медиаповестки

Ноутбук поддерживает два формальных режима работы:

1. `Работа с готовыми локальными артефактами`
2. `Полный локальный прогон`

Канонический pipeline состоит из пяти шагов:

1. очистка корпуса и нормализация даты, региона и текстового поля;
2. построение набора кандидатов с широким охватом статей;
3. построение или использование готовый story-level embeddings;
4. отбор экономических статей, присвоение тем BoE и расчёт регионального индекса;
5. построение итоговых таблиц, карт и графиков.

Во всех разделах ниже сначала дано содержательное объяснение этапа, а затем идут ячейки, которые либо запускают соответствующий скрипт, либо читают и проверяют его результаты.


In [ ]:
# Базовая инициализация notebook.
# Здесь мы:
# 1) подключаем библиотеки для чтения parquet/CSV и отображения результатов;
# 2) определяем корень проекта, даже если notebook открыт из папки notebooks/;
# 3) читаем settings.yaml и topic_profiles.yaml один раз в память;
# 4) собираем словарь paths со всеми рабочими путями пайплайна;
# 5) задаём переключатели режима: запускать ли cleaning/candidate stage и нужно ли
#    переиспользовать уже готовые embeddings.

from pathlib import Path
import json
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
import yaml
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / 'final_project').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'final_project'

CONFIG_PATH = PROJECT_ROOT / 'config' / 'settings.yaml'
TOPIC_PATH = PROJECT_ROOT / 'config' / 'topic_profiles.yaml'

with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    settings = yaml.safe_load(handle)
with TOPIC_PATH.open('r', encoding='utf-8') as handle:
    topic_profiles = yaml.safe_load(handle)['topics']

BOE_TOPICS = [topic['name'] for topic in topic_profiles]
paths = {key: PROJECT_ROOT / value for key, value in settings['paths'].items()}
PYTHON = str((PROJECT_ROOT.parent / '.venv' / 'Scripts' / 'python.exe').resolve())

RUN_CLEANING = False
RUN_CANDIDATES = False
USE_EXISTING_EMBEDDINGS = True
MODEL_SIZE = settings['models']['default_sentence_transformer_size']
RUN_MODE = settings['runtime']['run_mode']

def read_parquet_head(path, columns=None, n_rows=5000):
    parquet_file = pq.ParquetFile(path)
    first_batch = next(parquet_file.iter_batches(batch_size=n_rows, columns=columns))
    return first_batch.to_pandas()

def path_status(path):
    return 'exists' if Path(path).exists() else 'missing'

PROJECT_ROOT

In [ ]:
# Аудит конфигурации.
# Ячейка печатает весь settings.yaml
# Показывает с какими параметрами был выполнен прогон.

display(Markdown('## settings.yaml'))
display(Markdown(f'**run_mode:** `{settings["runtime"]["run_mode"]}`'))
print(CONFIG_PATH.read_text(encoding='utf-8'))

In [ ]:
# Аудит профилей тем Bank of England.
# Здесь выводится topic_profiles.yaml
# Для каждой темы в YAML есть name, description, keywords и seed_examples.

display(Markdown('## topic_profiles.yaml'))
print(TOPIC_PATH.read_text(encoding='utf-8'))

In [ ]:
# Сводка режима выполнения.
# Показывает какие шаги будут реально выполняться, а какие будут переиспользовать уже готовые артефакты.

display(Markdown('## Режим выполнения'))
mode_label = 'Работа с готовыми локальными артефактами' if USE_EXISTING_EMBEDDINGS else 'Полный локальный прогон'
display(Markdown(f'**Выбранный режим:** `{mode_label}`'))
display(Markdown(
    f'`RUN_CLEANING={RUN_CLEANING}`, `RUN_CANDIDATES={RUN_CANDIDATES}`, '
    f'`USE_EXISTING_EMBEDDINGS={USE_EXISTING_EMBEDDINGS}`, '
    f'`MODEL_SIZE={MODEL_SIZE}`, `RUN_MODE={RUN_MODE}`'
))
display(Markdown(
    f'Текущие артефакты: `cleaned={path_status(paths["cleaned_articles"])}`, '
    f'`candidate={path_status(paths["candidate_articles"])}`, '
    f'`lookup={path_status(paths["story_lookup"])}`, '
    f'`embeddings={path_status(paths["story_embeddings"])}`'
))

## 1. Очистка и проверка корпуса

На этом этапе сырые строки корпуса приводятся к рабочему формату. Шаг реализован в `scripts/01_clean_articles.py`.

Что именно делает cleaning stage:

- приводит две временные метки к единой дате наблюдения: по умолчанию берётся `tweet_date`, но если `article_date` существует и не расходится с `tweet_date` более чем на допустимый порог, то используется `article_date`;
- в поле `tweet_date` корректно обрабатывает случаи с несколькими датами, разделёнными `;`: берётся минимальная валидная дата;
- формирует итоговый регион как `main_LAD`, а если он пустой, то как `LAD`;
- очищает текст статьи очень простым, но жёстким правилом: `NaN -> пустая строка`, затем все последовательности пробельных символов схлопываются в один пробел и обрезаются по краям;
- удаляет строки без даты, без региона, с текстом короче порога `text_min_length` и с датами вне допустимого диапазона;
- создаёт `week` как понедельник соответствующей недели;
- создаёт `story_key` иерархически: `duplicate_group -> article_id -> unique_article_id`;
- создаёт `text_for_embedding` как первые `text_for_embedding_chars` символов `text_clean`;
- создаёт `dup_weight`, чтобы несколько дублирующих публикаций одной истории внутри одного `story_key × region × week` не раздували веса.

Именно после этого шага корпус становится пригодным для дальнейшей работы.

In [ ]:
# Запуск step 1.
# Если RUN_CLEANING=True, ноутбук вызывает 01_clean_articles.py.
# Скрипт читает raw articles.csv, чистит текст, нормализует даты и регионы,
# создаёт story_key / week / text_for_embedding / dup_weight и сохраняет cleaned_articles.parquet.
# Если RUN_CLEANING=False, используется уже существующий cleaned_articles.parquet.

if RUN_CLEANING:
    clean_run = subprocess.run(
        [PYTHON, 'scripts/01_clean_articles.py', '--config', str(CONFIG_PATH)],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=True,
    )
    print(clean_run.stdout)
    if clean_run.stderr.strip():
        print(clean_run.stderr)
else:
    print(f'Skipping step 1. Using existing file: {paths["cleaned_articles"]}')

In [ ]:
# Проверка результатов cleaning stage.
# Ячейка читает cleaning_report.json и первые строки cleaned_articles.parquet.
# Здесь можно увидеть:
# - сколько строк осталось после очистки;
# - сколько строк было отброшено как слишком короткие или как date outliers;
# - какой диапазон дат сохранился;
# - как выглядят итоговые поля date / week / region / story_key / dup_weight / text_clean.

with paths['cleaning_report'].open('r', encoding='utf-8') as handle:
    cleaning_report = json.load(handle)

summary = cleaning_report['summary']

cleaning_summary = pd.Series({
    'rows_seen': summary['rows_seen'],
    'rows_kept': summary['rows_kept'],
    'rows_dropped_short_text': summary['rows_dropped_short_text'],
    'rows_dropped_date_outlier': summary['rows_dropped_date_outlier'],
    'date_min': summary['clean_date_min'],
    'date_max': summary['clean_date_max'],
    'duplicate_story_groups': summary['duplicate_story_groups'],
}).to_frame('value')

display(cleaning_summary)

cleaned_head = read_parquet_head(
    paths['cleaned_articles'],
    columns=['unique_article_id', 'date', 'week', 'region', 'story_key', 'dup_weight', 'text_len', 'text_clean'],
    n_rows=5000,
)

display(cleaned_head.head(10))
display(cleaned_head['region'].value_counts().head(10).to_frame('rows'))
display(cleaned_head['text_len'].describe().to_frame('text_len'))

## 2. Построение набора кандидатов

На этом шаге строится набор кандидатов для semantic stage. Шаг реализован в `scripts/02_filter_candidates.py`.

Логика простая:

- по `text_clean` ищутся сильные экономические выражения из списка `strong_keywords`;
- по `text_clean` ищутся более широкие, менее надёжные сигналы из списка `weak_keywords`;
- по `resolved_url` ищутся тематические URL-секции из списка `url_terms`;
- статья попадает в candidates, если сработал хотя бы один из трёх сигналов.

Итоговые флаги:

- `strong_keyword_hit`
- `weak_keyword_hit`
- `url_section_hit`
- `candidate_source`

Поле `candidate_source` задаётся детерминированно по приоритету:

- `strong_keyword`
- `weak_keyword+url_section`
- `weak_keyword`
- `url_section`

Важно: этот шаг ещё не решает, является ли статья экономической. Он только строит компактный и более дешёвый для семантики набор кандидатов с высоким охватом.

In [ ]:
# Запуск step 2.
# Если RUN_CANDIDATES=True, notebook вызывает 02_filter_candidates.py.
# Скрипт читает cleaned_articles.parquet, добавляет keyword/url флаги,
# оставляет только строки-кандидаты и сохраняет candidate_articles.parquet.
# Если RUN_CANDIDATES=False, используется уже существующий candidate file.

if RUN_CANDIDATES:
    candidate_run = subprocess.run(
        [PYTHON, 'scripts/02_filter_candidates.py', '--config', str(CONFIG_PATH)],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=True,
    )
    print(candidate_run.stdout)
    if candidate_run.stderr.strip():
        print(candidate_run.stderr)
else:
    print(f'Skipping step 2. Using existing file: {paths["candidate_articles"]}')

In [ ]:
# Диагностика candidate stage.
# Ячейка читает preview candidate_articles.parquet и показывает,
# какими именно путями статьи попали в кандидаты.
# Это нужно для аудита recall/noise на дешёвом фильтре до запуска embedding model.

candidate_head = read_parquet_head(
    paths['candidate_articles'],
    columns=['unique_article_id', 'date', 'week', 'region', 'candidate_source', 'strong_keyword_hit', 'weak_keyword_hit', 'url_section_hit'],
    n_rows=5000,
)
candidate_source_distribution = candidate_head['candidate_source'].value_counts(dropna=False).rename_axis('candidate_source').to_frame('rows')

candidate_summary = pd.Series({
    'candidate_rows_in_preview': len(candidate_head),
    'share_strong_keyword': float(candidate_head['strong_keyword_hit'].mean()),
    'share_weak_keyword': float(candidate_head['weak_keyword_hit'].mean()),
    'share_url_section': float(candidate_head['url_section_hit'].mean()),
    'unique_regions_in_preview': int(candidate_head['region'].nunique()),
    'unique_weeks_in_preview': int(pd.to_datetime(candidate_head['week']).nunique()),
})

display(Markdown('### Сводка candidate gate'))
display(candidate_summary.to_frame('value'))
display(Markdown('### Структура candidate_source в preview'))
display(candidate_source_distribution)
display(Markdown('### Preview candidate rows без текстовых фрагментов'))
display(candidate_head.head(20))

## 3. Построение или использование готовых story embeddings

На этом шаге единицей анализа становится не строка со статьёй (article row), а `story_key`. Шаг реализован в `scripts/03_build_embeddings.py`.

Что именно происходит в режиме `build`:

- candidate-строки группируются по `story_key × region × week`, чтобы понять, где и когда каждая история присутствовала;
- для каждого `story_key` определяется основной `region × week` bucket: сначала по максимальному числу article rows, а при равенстве через детерминированный stable hash;
- считается `region_spread` — число уникальных регионов, где появилась история;
- в режиме `dev` не все истории проходят в semantic stage: выбирается до `embedding_story_cap` уникальных `story_key`;
- выборка в `dev` строится по уровням `strong` и `weak_url`, затем квоты распределяются пропорционально по `region × week`, а внутри бакета истории сортируются детерминированно по stable hash;
- после отбора для каждого `story_key` embedding считается ровно один раз по полю `text_for_embedding`.

Что делает режим `bootstrap-existing`:

- embeddings заново не считаются;
- из legacy story-level артефакта собирается новый `story_lookup.parquet`;
- это позволяет быстро продолжить scoring и построение индекса на уже готовых embeddings.

На выходе шага получаются два ключевых файла:

- `story_lookup.parquet`
- `story_embeddings.npy`

In [ ]:
# Запуск step 3.
# Ветка USE_EXISTING_EMBEDDINGS=True не пересчитывает тяжёлые embeddings,
# а только восстанавливает минимальный story_lookup через bootstrap-existing.
# Ветка USE_EXISTING_EMBEDDINGS=False строит embeddings с нуля:
# выбирает нужные story_key, кодирует text_for_embedding и сохраняет .npy + lookup.

if USE_EXISTING_EMBEDDINGS:
    embedding_cmd = [
        PYTHON,
        'scripts/03_build_embeddings.py',
        '--config',
        str(CONFIG_PATH),
        '--mode',
        'bootstrap-existing',
    ]
    step3_label = 'Работа с готовыми локальными артефактами'
else:
    embedding_cmd = [
        PYTHON,
        'scripts/03_build_embeddings.py',
        '--config',
        str(CONFIG_PATH),
        '--mode',
        'build',
        '--model-size',
        MODEL_SIZE,
        '--run-mode',
        RUN_MODE,
    ]
    step3_label = 'Полный локальный прогон'

embedding_run = subprocess.run(
    embedding_cmd,
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
)

display(Markdown(f'### Сводка step 3 (`{step3_label}`)'))
print(embedding_run.stdout)
if embedding_run.stderr.strip():
    print(embedding_run.stderr)

In [ ]:
# Проверка story-level артефактов.
# story_lookup связывает порядковый номер строки embedding matrix с конкретной историей.
# Поля region и week здесь не означают все регионы, которым принадлежит история: это её primary bucket 
# (где у этой истории больше всего публикаций), 
# выбранный детерминированно для sampling (отбора 120 тыс. историй) и lookup-таблицы.

story_lookup = pd.read_parquet(paths['story_lookup'])
story_embeddings = np.load(paths['story_embeddings'], mmap_mode='r')

embedding_summary = pd.Series({
    'story_lookup_rows': len(story_lookup),
    'embedding_rows': int(story_embeddings.shape[0]),
    'embedding_dim': int(story_embeddings.shape[1]),
    'story_embeddings_file_mb': round(paths['story_embeddings'].stat().st_size / (1024 ** 2), 2),
    'run_mode_in_lookup': ', '.join(sorted(story_lookup['run_mode'].astype(str).unique())),
}).to_frame('value')

display(embedding_summary)
display(story_lookup[['story_key', 'region', 'week', 'region_spread', 'run_mode']].head(15))
display(story_lookup['region'].value_counts().head(10).to_frame('stories'))

## 4. Economic gate, темы BoE и региональный индекс

Это главный содержательный шаг проекта. Он реализован в `scripts/04_score_and_build_index.py`.

### 4.1. Economic gate (отбор экономических статей)

Сначала для каждого `story_key` оценивается не тема BoE, а более широкая экономическая релевантность.

- story embedding сравнивается с positive econ anchors;
- тот же story embedding сравнивается с negative anchors, которые описывают типичные ложные срабатывания: спорт, криминал, развлечения, реклама, общие местные сюжеты;
- считаются `econ_positive_max`, `econ_negative_max` и `econ_margin = positive - negative`;
- история считается экономической, если она проходит основной semantic threshold либо более мягкий порог для `strong_keyword_hit`, но при этом не проигрывает negative anchors по правилу `negative_guard`.

### 4.2. Topic assignment

Затем для тех же story embeddings строятся эмбэдинги тем BoE.

Эмбэдинг темы задаётся не одним словом, а профилем из `topic_profiles.yaml`:

- `name`
- `description`
- `keywords`
- `seed_examples`

Для каждой истории считается косинусная близость ко всем 7 темам. После этого определяются:

- `top_topic`
- `top1_similarity`
- `top2_similarity`
- `topic_ambiguous`

Правило назначения темы:

- если `is_econ = False`, история получает `assigned_topic = unassigned`;
- если `is_econ = True` и `top1_similarity` проходит порог, история получает тему BoE;
- если `is_econ = True`, но порог темы не пройден, история получает `assigned_topic = other_econ`.

### 4.3. Возврат на article-level и построение индекса

Story-level результаты наследуются всеми статьями с тем же `story_key`. Затем для каждой article row строится

- `effective_weight = dup_weight / sqrt(region_spread)`.

После этого формируется панель `week × region × topic`, где считаются:

- `n_articles` как сумма `effective_weight` по теме;
- `total_econ_weight` как общий экономический вес региона и недели;
- `topic_share` как доля темы в экономическом потоке;
- `p_post` как сглаженная доля темы через empirical-Bayes prior;
- `baseline_share` как прошлое среднее по 8 неделям;
- `recent_share` как прошлое среднее по 4 неделям;
- `surprise = p_post - baseline_share`;
- `momentum = p_post - recent_share`;
- `mean_similarity` как средняя семантическая степень соответствия теме;
- `risk_score` как итоговая композиция `surprise_z`, `momentum_z` и `mean_similarity_z` с downweight по малому покрытию.

Подробная математическая версия всех переменных и формул вынесена в `RISK_INDEX_GUIDE.md`.

In [ ]:
# Запуск step 4.
# Скрипт читает story_lookup + story_embeddings, затем:
# 1) считает econ gate на уровне story_key;
# 2) назначает темы BoE или other_econ;
# 3) переносит story-level результаты обратно на статьи;
# 4) считает effective_weight;
# 5) строит regional_topic_index.parquet.
# Если USE_EXISTING_EMBEDDINGS=True, шаг повторно использует уже готовые эмбэдинги.

score_cmd = [
    PYTHON,
    'scripts/04_score_and_build_index.py',
    '--config',
    str(CONFIG_PATH),
    '--model-size',
    MODEL_SIZE,
]
if USE_EXISTING_EMBEDDINGS:
    score_cmd.append('--reuse-embeddings')

score_run = subprocess.run(
    score_cmd,
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
)

display(Markdown(
    f'### Сводка step 4 (`model_size={MODEL_SIZE}`, `reuse_embeddings={USE_EXISTING_EMBEDDINGS}`)'
))
print(score_run.stdout)
if score_run.stderr.strip():
    print(score_run.stderr)

In [ ]:
# Диагностика semantic scoring и индекса.
# Ячейка показывает:
# - сколько всего article rows получили scoring;
# - сколько из них признаны экономическими;
# - сколько попало в 7 тем BoE и сколько ушло в other_econ;
# - сколько регионов и недель вошло в панель индекса.

scored_articles = pd.read_parquet(paths['scored_articles'])
regional_index = pd.read_parquet(paths['regional_topic_index'])

scored_articles['week'] = pd.to_datetime(scored_articles['week'])
regional_index['week'] = pd.to_datetime(regional_index['week'])

econ_mask = scored_articles['is_econ']
scoring_summary = pd.Series({
    'scored_articles': len(scored_articles),
    'economic_articles': int(econ_mask.sum()),
    'boe_topic_articles': int(scored_articles['assigned_topic'].isin(BOE_TOPICS).sum()),
    'other_econ_articles': int(scored_articles['assigned_topic'].eq('other_econ').sum()),
    'regions_in_index': int(regional_index['region'].nunique()),
    'weeks_in_index': int(regional_index['week'].nunique()),
    'other_econ_share_among_econ_articles': float(scored_articles.loc[econ_mask, 'assigned_topic'].eq('other_econ').mean()),
})
topic_distribution = scored_articles.loc[econ_mask, 'assigned_topic'].value_counts().rename_axis('assigned_topic').to_frame('rows')

display(Markdown('### Сводка semantic scoring'))
display(scoring_summary.to_frame('value'))
display(Markdown('### Распределение assigned_topic среди экономических статей'))
display(topic_distribution.head(12))

In [ ]:
# Быстрый просмотр strongest region-topic pairs по итоговому risk_score.
# Это raw view по panel output, ещё до финальной витрины step 5.

display(Markdown('### Top region-topic pairs'))
display(regional_index.sort_values('risk_score', ascending=False).head(20))

## 5. Итоговые таблицы и графики

Финальный шаг реализован в `scripts/05_make_outputs.py`. Он работает с `regional_topic_index.parquet` и `scored_articles.parquet`.

Что делает этот шаг:

- выбирает reference week как последнюю активную неделю 2022 года по темам BoE, а если таких недель нет, то последнюю активную неделю 2021 года;
- выбирает одну рабочую тему для детального показа на тематической карте и на тематическом графике трендов;
- строит агрегированную таблицу `hot_regions_topics.csv`;
- формирует таблицу `representative_snippets.csv` для сильнейших region-topic сочетаний reference week;
- загружает геоданные LAD boundaries из `data/external/uk_lad_2023_buc.geojson` и строит карты;
- строит тренды по выбранной теме и по совокупному региональному риску.


In [ ]:
# Запуск step 5.
# Скрипт создаёт финальные таблицы и графики на основе уже готовых scored_articles и regional_topic_index.

output_run = subprocess.run(
    [PYTHON, 'scripts/05_make_outputs.py', '--config', str(CONFIG_PATH)],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(output_run.stdout)
if output_run.stderr.strip():
    print(output_run.stderr)

In [ ]:
# Чтение итоговой агрегированной таблицы hot pairs.
# Эта ячейка показывает основные region-topic результаты step 5.
# Если локально существует таблица representative_snippets.csv, notebook дополнительно выводит её верхние строки.

hot_pairs = pd.read_csv(paths['hot_regions_topics'])

reference_week = 'no active BoE week found'
if not hot_pairs.empty:
    reference_week = pd.to_datetime(hot_pairs['week']).max().date().isoformat()

display(Markdown(f'### Reference week used for latest outputs: `{reference_week}`'))
display(hot_pairs.head(20))

snippets_path = paths.get('representative_snippets')
display(Markdown('### Representative snippets for latest hot pairs'))
if snippets_path is not None and snippets_path.exists():
    snippets = pd.read_csv(snippets_path)
    display(snippets.head(15))
else:
    display(Markdown('Файл `representative_snippets.csv` не найден. Он создаётся на step 5 при наличии локальных scored_articles.'))


In [ ]:
# Отрисовка всех итоговых figures, пути к которым заданы в settings.yaml.
# Ячейка заново перечитывает конфиг, чтобы не зависеть от старого состояния kernel,
# и затем по очереди показывает карты и трендовые графики, если соответствующие файлы существуют.

with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    latest_settings = yaml.safe_load(handle)

latest_paths = {key: PROJECT_ROOT / value for key, value in latest_settings['paths'].items()}

figure_specs = [
    ('### Карта: reference week', 'latest_heatmap'),
    ('### Карта: среднее за декабрь 2022', 'monthly_heatmap'),
    ('### Карта: общий top-risk по регионам', 'overall_risk_map'),
    ('### Карта: доминирующая тема по регионам', 'dominant_topic_map'),
    ('### График трендов: showcase topic', 'hot_topic_trends'),
    ('### График трендов: совокупный региональный риск', 'overall_region_trends'),
]

for title, key in figure_specs:
    path = latest_paths.get(key)
    display(Markdown(title))
    if path is None:
        display(Markdown(f'Path `{key}` is missing from `settings.yaml`.'))
        continue
    if not path.exists():
        display(Markdown(f'File not found: `{path}`'))
        continue
    display(Image(filename=str(path)))